In [10]:
from lxml import html
import json
import requests
import os
from time import sleep
from random import randint
from dataclasses import dataclass, field
from typing import List, Dict
from time import asctime
from datetime import date
import sqlite3


In [2]:
@dataclass
class ID:
    last_name: str
    first_name: str
    middle_name: str
    
    # Add Address, Phone etc.
    
    def __str__(self):
        return f"'{self.last_name}', '{self.first_name}' '{self.middle_name}'"
   
rendier = ID('Allison', 'Cody', 'Michael')

print(rendier)

'Allison', 'Cody' 'Michael'


In [3]:
rendier

ID(last_name='Allison', first_name='Cody', middle_name='Michael')

In [4]:
@dataclass
class Wallet:
    identity: List[ID]
    cash: float = 0.0
    
    # Add cards eventually
    
    def __post_init__(self):
        self.receipts = []
        self.separator = "-" * 16
    
    def __str__(self):
        return f"{self.identity}. ${self.cash:.2f}"
    
    def add_money(self, amount, symbol, volume):
        
        transaction = [symbol, self.cash, amount, self.cash + amount, volume]
        self.cash += amount
        self.add_receipt(transaction, "+")
    
    def spend_money(self, amount, symbol, volume):
        
        transaction = [symbol, self.cash, amount, self.cash - amount, volume]
        self.cash -= amount
        self.add_receipt(transaction, "-")
        
    
    def add_receipt(self, transaction, sign):
        timestr = asctime()
        symbol = transaction[0]
        previous = transaction[1]
        amount = transaction[2]
        balance = transaction[3]
        volume = transaction[4]
            
        self.receipt = Receipt(self.identity, timestr, symbol, previous, sign, amount, balance, volume)
        
        self.receipts.append(self.receipt)
        pass
    
    def z_print(self):
        
        pass

In [5]:
@dataclass
class Receipt:
    identity: List[ID]
    time_date: str
    symbol: str
    previous: float
    sign: str
    amount: float
    balance: float
    volume: int
    
    
    def __post_init__(self):
        self.separator = "-" * 16
        print(f"RECEIPT:\n{self}")
    
    def __str__(self):
        return f"{self.time_date}\n{self.symbol}\n${self.previous:15.2f}\n{self.sign}{self.amount:15.2f}\n{self.separator}\n>{self.balance:15.2f}\nVolume:\n{self.volume:16}"
 

In [6]:
timestamp = asctime()
print(timestamp)


Thu Jan 17 17:27:43 2019


In [7]:
#mine = Wallet('Allison', 'Cody', 'Michael', 100.00)
mine = Wallet(rendier, 100.00)

print(mine)

'Allison', 'Cody' 'Michael'. $100.00


In [8]:
mine

Wallet(identity=ID(last_name='Allison', first_name='Cody', middle_name='Michael'), cash=100.0)

In [9]:
mine.add_money(145.60, 'AAPL', 35000)

RECEIPT:
Thu Jan 17 17:27:46 2019
AAPL
$         100.00
+         145.60
----------------
>         245.60
Volume:
           35000


In [10]:
print(mine.receipts, mine)


[Receipt(identity=ID(last_name='Allison', first_name='Cody', middle_name='Michael'), time_date='Thu Jan 17 17:27:46 2019', symbol='AAPL', previous=100.0, sign='+', amount=145.6, balance=245.6, volume=35000)] 'Allison', 'Cody' 'Michael'. $245.60


In [11]:
mine.spend_money(45.60, 'AAPL', 5000)

RECEIPT:
Thu Jan 17 17:27:49 2019
AAPL
$         245.60
-          45.60
----------------
>         200.00
Volume:
            5000


In [12]:
@dataclass
class Stock:
    board: str
    symbol: str
    volume: int
    
    stock_data: dict = None
    name: str = ""
    closing_price: float = 0.0
    closing_date: str = ""
    volume_price: float = 0.0
    
    
    
    def __post_init__(self):
        if self.board.lower() == 'nasdaq':
            self.update_nasdaq_data()
    
    def __str__(self):
        return f"{self.board.upper()} Stock: {self.symbol}\n{self.name}\n{self.closing_date}\nClose: {self.closing_price}\nVolume: {self.volume}\nVolume Price: {self.volume_price}"

    
    def update_nasdaq_data(self):
        self.stock_data = self.parse_nasdaq_stock(self.symbol)
        print("Finished Parsing")
        print(self.stock_data)
        self.name = self.stock_data['company_name']
        self.closing_price = float(self.stock_data['close_price'].replace("$", "").replace(" ", ""))
        self.closing_date = self.stock_data['close_date']
        print("CLOSINGPRICE:", self.closing_price)
        print("VOLUME:", self.volume)
        self.volume_price = self.volume * self.closing_price
       
    def parse_nasdaq_stock(self, ticker):
        """
        Grab financial data from NASDAQ page

        Args:
          ticker (str): Stock symbol

        Returns:
          dict: Scraped data
        """
        key_stock_dict = {}
        headers = {
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
            "Accept-Encoding": "gzip, deflate",
            "Accept-Language": "en-GB,en;q=0.9,en-US;q=0.8,ml;q=0.7",
            "Connection": "keep-alive",
            "Host": "www.nasdaq.com",
            "Referer": "http://www.nasdaq.com",
            "Upgrade-Insecure-Requests": "1",
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/64.0.3282.119 Safari/537.36"
        }
        
        # Retrying for failed request
        for retries in range(5):
            try:
                url = "http://www.nasdaq.com/symbol/%s" % (ticker)
                response = requests.get(url, headers=headers, verify=True)
                
                if response.status_code != 200:
                    raise ValueError("Invalid Response Received From Webserver")
                
                print("Parsing %s" % (url))
                # Adding random delay
                sleep(randint(1, 3))
                parser = html.fromstring(response.text)
                xpath_head = "//div[@id='qwidget_pageheader']//h1//text()"
                xpath_key_stock_table = '//div[@class="row overview-results relativeP"]//div[contains(@class,"table-table")]/div'
                xpath_open_price = '//b[contains(text(),"Open Price:")]/following-sibling::span/text()'
                xpath_open_date = '//b[contains(text(),"Open Date:")]/following-sibling::span/text()'
                xpath_close_price = '//b[contains(text(),"Close Price:")]/following-sibling::span/text()'
                xpath_close_date = '//b[contains(text(),"Close Date:")]/following-sibling::span/text()'
                xpath_key = './/div[@class="table-cell"]/b/text()'
                xpath_value = './/div[@class="table-cell"]/text()'
                
                raw_name = parser.xpath(xpath_head)
                key_stock_table = parser.xpath(xpath_key_stock_table)
                raw_open_price = parser.xpath(xpath_open_price)
                raw_open_date = parser.xpath(xpath_open_date)
                raw_close_price = parser.xpath(xpath_close_price)
                raw_close_date = parser.xpath(xpath_close_date)
                
                company_name = raw_name[0].replace("Common Stock Quote & Summary Data", "").strip() if raw_name else ''
                open_price = raw_open_price[0].strip() if raw_open_price else None
                open_date = raw_open_date[0].strip() if raw_open_date else None
                close_price = raw_close_price[0].strip() if raw_close_price else None
                close_date = raw_close_date[0].strip() if raw_close_date else None
                
                # Grabbing ans cleaning keystock data
                for i in key_stock_table:
                    key = i.xpath(xpath_key)
                    value = i.xpath(xpath_value)
                    key = ''.join(key).strip()
                    value = ' '.join(''.join(value).split())
                    key_stock_dict[key] = value
                
                nasdaq_data = {
                    
                    "company_name": company_name,
                    "ticker": ticker,
                    "url": url,
                    "open price": open_price,
                    "open_date": open_date,
                    "close_price": close_price,
                    "close_date": close_date,
                    "key_stock_data": key_stock_dict
                }
                return nasdaq_data
            
            except Exception as e:
                print("Failed to process the request, Exception:%s" % (e))

In [13]:
AAPL = Stock('nasdaq', 'AAPL', 35000)

Parsing http://www.nasdaq.com/symbol/AAPL
Finished Parsing
{'company_name': 'Apple Inc. Common Stock (AAPL) Quote & Summary Data', 'ticker': 'AAPL', 'url': 'http://www.nasdaq.com/symbol/AAPL', 'open price': '$\xa0154.22', 'open_date': 'Jan. 17, 2019', 'close_price': '$\xa0154.94', 'close_date': 'Jan. 16, 2019', 'key_stock_data': {'Best Bid / Ask': '$ 155.65 / $ 155.80', '1 Year Target': '197.5', "Today's High / Low": '$ 157.66 / $ 153.26', 'Share Volume': '29,633,411', '50 Day Avg. Daily Volume': '45,699,328', 'Previous Close': '$ 154.94', '52 Week High / Low': '$ 233.47 / $ 142', 'Market Cap': '737,187,095,580', 'P/E Ratio': '13.13', 'Forward P/E (1y)': '12.80', 'Earnings Per Share (EPS)': '$ 11.87', 'Annualized Dividend': '$ 2.92', 'Ex Dividend Date': '11/8/2018', 'Dividend Payment Date': '11/15/2018', 'Current Yield': '1.91 %', 'Beta': '1.02'}}
CLOSINGPRICE: 154.94
VOLUME: 35000


In [14]:
print(AAPL)

NASDAQ Stock: AAPL
Apple Inc. Common Stock (AAPL) Quote & Summary Data
Jan. 16, 2019
Close: 154.94
Volume: 35000
Volume Price: 5422900.0


In [338]:
AAPL

Stock(board='nasdaq', symbol='AAPL', volume=35000, stock_data={'company_name': 'Apple Inc. Common Stock (AAPL) Quote & Summary Data', 'ticker': 'AAPL', 'url': 'http://www.nasdaq.com/symbol/AAPL', 'open price': '$\xa0153.05', 'open_date': 'Jan. 16, 2019', 'close_price': '$\xa0154.94', 'close_date': 'Jan. 16, 2019', 'key_stock_data': {'Best Bid / Ask': 'N/A / N/A', '1 Year Target': '197.5', "Today's High / Low": '$ 155.88 / $ 153', 'Share Volume': '30,569,706', '50 Day Avg. Daily Volume': '46,245,190', 'Previous Close': '$ 153.07', '52 Week High / Low': '$ 233.47 / $ 142', 'Market Cap': '732,835,676,820', 'P/E Ratio': '13.05', 'Forward P/E (1y)': '12.65', 'Earnings Per Share (EPS)': '$ 11.87', 'Annualized Dividend': '$ 2.92', 'Ex Dividend Date': '11/8/2018', 'Dividend Payment Date': '11/15/2018', 'Current Yield': '1.95 %', 'Beta': '1.02'}}, name='Apple Inc. Common Stock (AAPL) Quote & Summary Data', closing_price=154.94, closing_date='Jan. 16, 2019', volume_price=5422900.0)

In [375]:
@dataclass
class Portfolio:
    identity: List[ID]
    wallet: List[Wallet]
    nasdaqstocks: list = None # {stock: # of shares}
    nysestocks: list = None
    amexstocks: list = None
    
    
    def __post_init__(self):
        self.nasdaqstocks = []
        self.nysestocks = []
        self.amexstocks = []
        pass
    
    def __str__(self):
        return f"{self.identity}\n{self.wallet.cash}\nNASDAQ:\n{self.nasdaqstocks}"
    
    def add_nasdaq(self, symbol, volume):
        
        #stock = Stock('nasdaq', symbol, volume)
        #print("STOCK:", stock)
        #self.nasdaqstocks.append(stock)
        
        code = f"{symbol.upper()} = Stock('nasdaq', '{symbol}', {volume})"
        print("Add Nasdaq Code 1:", code)
        #print(AAPL)
        exec(code)
        
        code = f"self.nasdaqstocks.append([{symbol}, {volume}])"
        print("Add Nasdaq Code 2:", code)
        exec(code)

        pass

In [376]:
Watch = Portfolio(rendier, mine)

In [377]:
Watch.add_nasdaq("AAPL", 35000)

Add Nasdaq Code 1: AAPL = Stock('nasdaq', 'AAPL', 35000)
Parsing http://www.nasdaq.com/symbol/AAPL
Finished Parsing
CLOSINGPRICE: 154.94
VOLUME: 35000
Add Nasdaq Code 2: self.nasdaqstocks.append([AAPL, 35000])


In [378]:
Watch.nasdaqstocks[0][0].symbol

'AAPL'

In [379]:
print(Watch)

'Allison', 'Cody' 'Michael'
200.0
NASDAQ:
[[Stock(board='nasdaq', symbol='AAPL', volume=35000, stock_data={'company_name': 'Apple Inc. Common Stock (AAPL) Quote & Summary Data', 'ticker': 'AAPL', 'url': 'http://www.nasdaq.com/symbol/AAPL', 'open price': '$\xa0153.05', 'open_date': 'Jan. 16, 2019', 'close_price': '$\xa0154.94', 'close_date': 'Jan. 16, 2019', 'key_stock_data': {'Best Bid / Ask': 'N/A / N/A', '1 Year Target': '197.5', "Today's High / Low": '$ 155.88 / $ 153', 'Share Volume': '30,569,706', '50 Day Avg. Daily Volume': '46,245,190', 'Previous Close': '$ 153.07', '52 Week High / Low': '$ 233.47 / $ 142', 'Market Cap': '732,835,676,820', 'P/E Ratio': '13.05', 'Forward P/E (1y)': '12.65', 'Earnings Per Share (EPS)': '$ 11.87', 'Annualized Dividend': '$ 2.92', 'Ex Dividend Date': '11/8/2018', 'Dividend Payment Date': '11/15/2018', 'Current Yield': '1.95 %', 'Beta': '1.02'}}, name='Apple Inc. Common Stock (AAPL) Quote & Summary Data', closing_price=154.94, closing_date='Jan. 16, 

In [386]:
GPL = Stock('nasdaq', 'GPL', 0)

Parsing http://www.nasdaq.com/symbol/GPL
Finished Parsing
{'company_name': 'Great Panther Silver Limited Ordinary Shares (Canada) (GPL) Quote & Summary Data', 'ticker': 'GPL', 'url': 'http://www.nasdaq.com/symbol/GPL', 'open price': None, 'open_date': None, 'close_price': None, 'close_date': None, 'key_stock_data': {'1 Year Target': '1.75', "Today's High / Low": '$ 0.6496 / $ 0.6205', 'Share Volume': '412,923', '90 Day Avg. Daily Volume': '553,192', 'Previous Close': '$ 0.646', '52 Week High / Low': '$ 1.45 / $ 0.535', 'Market Cap': '105,914,211', 'Annualized Dividend': 'N/A', 'Ex Dividend Date': 'N/A', 'Dividend Payment Date': 'N/A', 'Current Yield': '0 %', 'Beta': '-0.02'}}


AttributeError: 'NoneType' object has no attribute 'replace'

In [3]:
def update_stock_data():

    AMEX = {}
    NYSE = {}
    Nasdaq = {}
    TheBoards = {}
    
    boards = ["AMEX", "NYSE", "Nasdaq"]
    headings = ['Name', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Net Chg', '% Chg', 'Volume', '52 Wk High', '52 Wk Low', 'Div', 'Yield', 'P/E', 'YTD % Chg']
    
    for board in boards:
        print(f"Updating {board} Data\n")
        
        today = date.today()
        filedate = today.strftime("%y%m%d")
        url = f"http://www.wsj.com/public/resources/documents/{board}.csv"
        try:
            file = requests.get(url)# Insert Try Except here TODO

        except ConnectionError:
            print("There is no connection to the internet available.\nPlease try again after a connection is established")
            break

        page = file.text

        lines = page.split("\n")

        lines = lines[4:-1]
        
        rows = []
        
        for line in lines:
            start = line.find('"')
            stop = line.rfind('"') + 1
            changethis = line[start: stop]
            tothis = changethis.replace(",", "").replace('"', '')
            rows.append(line.replace(changethis, tothis).split(","))
        


        for row in rows:
            entry = dict(zip(headings, row))
            for item in entry:
                value = entry[item]
                try:
                    entry[item] = float(value)
                except ValueError:
                    pass
                
            code = f"{board}[row[1]] = entry"
            exec(code)
            
        code = f"TheBoards['{board}'] = {board}"
        exec(code)

    
    
    with open(f"{filedate}-TheBoards.json", 'w') as file:
        code = f"file.write(json.dumps({TheBoards}))"
        exec(code)
        file.close()

In [4]:
print(os.getcwd())

/home/rendier/Ptolemy/Phaleron/PennyWatcher


In [7]:
update_stock_data()

Updating AMEX Data

Updating NYSE Data

Updating Nasdaq Data



In [8]:
file = open(f'{date.today():%y%m%d}-TheBoards.json', 'r')
TheBoards = json.load(file)
file.close()

In [9]:
print(TheBoards['NYSE']['ZYME'])

KeyError: 'ZYME'